In [38]:
import toml
import message_ix
import ixmp
import pandas as pd
from ixmp import Scenario
from pathlib import Path
#from model.setup import setup_scenario # ! Deprecated implementation

from message_ix_models.util import (
    broadcast,
    make_matched_dfs,
    package_data_path,
    same_node,
    same_time,
)
from model.utils import(map_yv_ya_lt)
from message_ix import make_df

In [39]:
scenario_name = 'baseline'

In [40]:
platform, msg, msg_path = setup_scenario(scenario_name)

Sheet name: type_tec
sets type_tec
Sheet name: cat_tec
sets cat_tec
Sheet name: shares
sets shares
Sheet name: map_shares_commodity_share
sets map_shares_commodity_share
Sheet name: map_shares_commodity_total
sets map_shares_commodity_total
Sheet name: share_commodity_lo
par share_commodity_lo
['share_low_carbon' 'share_electric_transport' 'share_h2_rc' 'share_h2_i'
 'share_h2_fc_i' 'share_h2_fc_RC' 'share_h2_trp']


In [48]:
msg.remove_solution()

In [37]:
msg.check_out()

RuntimeError: unhandled Java exception: There was a problem retrieving the status for run id '2'!

In [ ]:
msg.par('')

In [ ]:
'''Game plan:
    - Look at supply side stuff for Mo? Done but idk if at the level he wants, chat
    - Add a hydrogen storage technology to the model for one that stores gaseous hydrogen and one that stores liquid hydrogen. Dont need to specify the size for now. In future it could be interesting to define geological potential.
    - You will need to figure out discharge rate etc
    - This will give you a sense of how to create a function to create the add tech function so that you can just defie inputs and run
    - Add other techs after this. Start with ATRs and go from there'''

In [13]:
# Select provinces automatically, if they are not defined in the input yaml
provinces = msg.set("node").to_list()[2:]

In [29]:
msg.add_set("technology", "h2_ATR_CCS")  #remove the caps for future...

# Define the time range for year_vtg and year_act
years = msg.set("year").to_list()[14:]
first_year = msg.firstmodelyear


<class 'int'>


In [19]:
msg.par("input").columns

Index(['node_loc', 'technology', 'year_vtg', 'year_act', 'mode', 'node_origin',
       'commodity', 'level', 'time', 'time_origin', 'value', 'unit'],
      dtype='object')

In [30]:
df = make_df("input", 
    node_loc=provinces, 
    technology="h2_ATR_CCS", 
    mode="M1",
    commodity="gas",
    level="secondary",
    time="year",
    time_origin="year",
    value=1,
    unit="-"
    ).pipe(same_node).pipe(same_time)
#print(df)

In [32]:
df = make_df("input", 
    node_loc=provinces, 
    technology="h2_ATR_CCS", 
    mode="M1",
    commodity="gas",
    level="secondary",
    time="year",
    time_origin="year",
    value=1,
    unit="-"
    ).pipe(broadcast,map_yv_ya_lt(
                            years, 40, first_year
                        ),
        time="year",
    ).pipe(same_node).pipe(same_time)

ValueError: Dimension time was not empty

                  node_loc  technology  year_vtg  year_act mode node_origin  \
0                  Alberta  h2_ATR_CCS      2020      2020   M1        None   
1          BritishColumbia  h2_ATR_CCS      2020      2020   M1        None   
2                 Manitoba  h2_ATR_CCS      2020      2020   M1        None   
3             NewBrunswick  h2_ATR_CCS      2020      2020   M1        None   
4  NewfoundlandandLabrador  h2_ATR_CCS      2020      2020   M1        None   

  commodity      level  time time_origin  value unit  
0       gas  secondary  year        year      1    -  
1       gas  secondary  year        year      1    -  
2       gas  secondary  year        year      1    -  
3       gas  secondary  year        year      1    -  
4       gas  secondary  year        year      1    -  

In [50]:

# Adding input from a commodity to the technology
for year_vtg in years:
    for year_act in years:
        if year_vtg >= year_act:
            msg.add_par("input", {
                "node_loc": provinces,
                "node_origin": provinces,
                "technology": "h2_ATR_CCS",
                "year_vtg": year_vtg,
                "year_act": year_act,
                "mode": "M1",
                "commodity": "gas",
                "level": "secondary",
                "time": "year",
                "time_origin": "year",
                "value": 1.2,
                "unit": "GWa"  # Ensure units are consistent with your data
            })

            msg.add_par("input", {
                "node_loc": provinces,
                "node_origin": provinces,
                "technology": "h2_ATR_CCS",
                "year_vtg": year_vtg,
                "year_act": year_act,
                "mode": "M1",
                "commodity": "electr",
                "level": "secondary",
                "time": "year",
                "time_origin": "year",
                "value": 0.05,
                "unit": "GWa"
            })

# Adding output to a commodity from the technology
for year_vtg in years:
    for year_act in years:
        if year_vtg >= year_act:
            msg.add_par("output", {
                "node_loc": provinces,
                "node_dest": provinces,
                "technology": "h2_ATR_CCS",
                "year_vtg": year_vtg,
                "year_act": year_act,
                "mode": "M1",
                "commodity": "hydrogen",
                "level": "secondary",
                "time": "year",
                "time_dest": "year",
                "value": 1,
                "unit": "GWa"
            })

# Adding investment cost
for year_vtg in years:
    msg.add_par("inv_cost", {
        "node_loc": provinces,
        "technology": "h2_ATR_CCS",
        "year_vtg": year_vtg,
        "value": 1298,
        "unit": "USD/kW"
    })

# Adding fixed cost
for year_vtg in years:
    for year_act in years:
        if year_vtg >= year_act:
            msg.add_par("fix_cost", {
                "node_loc": provinces,
                "technology": "h2_ATR_CCS",
                "year_vtg": year_vtg,
                "year_act": year_act,
                "value": 28.87,
                "unit": "USD/kW"
            })

# Adding variable cost
for year_vtg in years:
    for year_act in years:
        if year_vtg >= year_act:
            msg.add_par("var_cost", {
                "node_loc": provinces,
                "technology": "h2_ATR_CCS",
                "year_vtg": year_vtg,
                "year_act": year_act,
                "mode": "M1",
                "time": "year",
                "value": 0, #leaving as 0 for now based on SMR assumption. However I have a value if need be, just seems far too expensive...
                "unit": "USD/kW"
            })

# Adding technical lifetime
for year_vtg in years:
    msg.add_par("technical_lifetime", {
        "node_loc": provinces,
        "technology": "h2_ATR_CCS",
        "year_vtg": year_vtg,
        "value": 40,
        "unit": "year"
    })

# Adding construction time
for year_vtg in years:
    msg.add_par("construction_time", {
        "node_loc": provinces,
        "technology": "h2_ATR_CCS",
        "year_vtg": year_vtg,
        "value": 3,
        "unit": "year"
    })

# Adding capacity factor
for year_vtg in years:
    for year_act in years:
        if year_vtg >= year_act:
            msg.add_par("capacity_factor", {
                "node_loc": provinces,
                "technology": "h2_ATR_CCS",
                "year_vtg": year_vtg,
                "year_act": year_act,
                "time": "year",
                "value": 0.9,
            })

# Adding emission factor
for year_vtg in years:
    for year_act in years:
        if year_vtg >= year_act:
            msg.add_par("emission_factor", {
                "node_loc": provinces,
                "technology": "h2_ATR_CCS",
                "year_vtg": year_vtg,
                "year_act": year_act,
                "mode": "M1",
                "emission": "CO2",
                "value": 0,  #How is this modelled???
            })

In [26]:
msg.add_set("technology", "h2_stor_geo")  #does this make sense as the model doesnt investigate curtailment?

# Define the time range for year_vtg and year_act
years = range(2025, 2061, 5)

# Adding input from a commodity to the technology
for year_vtg in years:
    for year_act in years:
        if year_vtg >= year_act:
            msg.add_par("input", {
                "node_loc": provinces,
                "node_origin": provinces,
                "technology": "h2_stor_geo",
                "year_vtg": year_vtg,
                "year_act": year_act,
                "mode": "M1",
                "commodity": "hydrogen",
                "level": "secondary",
                "time": "year",
                "time_origin": "year",
                "value": 0.00381,
                "unit": "GWa"  # Ensure units are consistent with your data
            })

            msg.add_par("input", {
                "node_loc": provinces,
                "node_origin": provinces,
                "technology": "h2_stor_geo",
                "year_vtg": year_vtg,
                "year_act": year_act,
                "mode": "M1",
                "commodity": "electr",
                "level": "secondary",
                "time": "year",
                "time_origin": "year",
                "value": 0.000114,
                "unit": "GWa"
            })

# Adding output to a commodity from the technology
for year_vtg in years:
    for year_act in years:
        if year_vtg >= year_act:
            msg.add_par("output", {
                "node_loc": provinces,
                "node_dest": provinces,
                "technology": "h2_stor_geo",
                "year_vtg": year_vtg,
                "year_act": year_act,
                "mode": "M1",
                "commodity": "hydrogen",
                "level": "secondary",
                "time": "year",
                "time_dest": "year",
                "value": 0.0036195,
                "unit": "GWa"
            })

# Adding investment cost
for year_vtg in years:
    msg.add_par("inv_cost", {
        "node_loc": provinces,
        "technology": "h2_stor_geo",
        "year_vtg": year_vtg,
        "value": 422.57,
        "unit": "USD/kW"
    })

# Adding fixed cost
for year_vtg in years:
    for year_act in years:
        if year_vtg >= year_act:
            msg.add_par("fix_cost", {
                "node_loc": provinces,
                "technology": "h2_stor_geo",
                "year_vtg": year_vtg,
                "year_act": year_act,
                "value": 12.67,
                "unit": "USD/kW"
            })

# Adding variable cost
for year_vtg in years:
    for year_act in years:
        if year_vtg >= year_act:
            msg.add_par("var_cost", {
                "node_loc": provinces,
                "technology": "h2_stor_geo",
                "year_vtg": year_vtg,
                "year_act": year_act,
                "mode": "M1",
                "time": "year",
                "value": 0,
                "unit": "USD/kW"
            })

# Adding technical lifetime
for year_vtg in years:
    msg.add_par("technical_lifetime", {
        "node_loc": provinces,
        "technology": "h2_stor_geo",
        "year_vtg": year_vtg,
        "value": 40,
        "unit": "year"
    })

# Adding construction time
for year_vtg in years:
    msg.add_par("construction_time", {
        "node_loc": provinces,
        "technology": "h2_stor_geo",
        "year_vtg": year_vtg,
        "value": 3,
        "unit": "year"
    })

# Adding capacity factor
for year_vtg in years:
    for year_act in years:
        if year_vtg >= year_act:
            msg.add_par("capacity_factor", {
                "node_loc": provinces,
                "technology": "h2_stor_geo",
                "year_vtg": year_vtg,
                "year_act": year_act,
                "time": "year",
                "value": 0.8,
            })

# Adding emission factor
for year_vtg in years:
    for year_act in years:
        if year_vtg >= year_act:
            msg.add_par("emission_factor", {
                "node_loc": provinces,
                "technology": "h2_stor_geo",
                "year_vtg": year_vtg,
                "year_act": year_act,
                "mode": "M1",
                "emission": "CO2",
                "value": 0,
            })



In [27]:
msg.par("input", {"technology": "h2_stor_geo"})

,node_loc,technology,year_vtg,year_act,mode,node_origin,commodity,level,time,time_origin,value,unit
0,Alberta,h2_stor_geo,2025,2060,M1,Alberta,hydrogen,secondary,year,year,0.003810,GWa
1,BritishColumbia,h2_stor_geo,2025,2060,M1,BritishColumbia,hydrogen,secondary,year,year,0.003810,GWa
2,Manitoba,h2_stor_geo,2025,2060,M1,Manitoba,hydrogen,secondary,year,year,0.003810,GWa
3,NewBrunswick,h2_stor_geo,2025,2060,M1,NewBrunswick,hydrogen,secondary,year,year,0.003810,GWa
4,NewfoundlandandLabrador,h2_stor_geo,2025,2060,M1,NewfoundlandandLabrador,hydrogen,secondary,year,year,0.003810,GWa
...,...,...,...,...,...,...,...,...,...,...,...,...
957,Ontario,h2_stor_geo,2060,2060,M1,Ontario,electr,secondary,year,year,0.000114,GWa
958,PrinceEdwardIsland,h2_stor_geo,2060,2060,M1,PrinceEdwardIsland,electr,secondary,year,year,0.000114,GWa
959,Quebec,h2_stor_geo,2060,2060,M1,Quebec,electr,secondary,year,year,0.000114,GWa
960,Saskatchewan,h2_stor_geo,2060,2060,M1,Saskatchewan,electr,secondary,year,year,0.000114,GWa


In [40]:
# Define the time slices for each month
time_slices = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

# Add time slices to the model
msg.add_set("time", time_slices)

# Define the temporal level (e.g., year as the overarching level)
msg.add_set("lvl_temporal", "year")

# Add the time_order, ensuring the correct sequence
time_order_df = pd.DataFrame({
    "lvl_temporal": ["year"] * 12,
    "time": time_slices,
    "value": list(range(1, 13))  # Sequence from 1 to 12
})
msg.add_par("time_order", time_order_df)

# Optional: Set the fraction of time each slice represents if needed (e.g., each month is ~1/12 of a year)
time_duration_df = pd.DataFrame({
    "lvl_temporal": ["year"] * 12,
    "time": time_slices,
    "value": [1/12] * 12
})
msg.add_par("duration_time", time_duration_df)

# Create an empty list to hold the DataFrame rows
rows = []

# Loop through each province and time slice to create the rows
for province in provinces:
    for time_slice in time_slices:
        rows.append({
            "node": province,  # Correct column name
            "technology": "h2_stor_geo",
            "commodity": "hydrogen",
            "mode": "M1",
            "level":"secondary",
            "year": year_act,
            "time": time_slice,
            "value": 0.05  # 5% loss per month
        })

# Convert rows list to DataFrame
storage_self_discharge_df = pd.DataFrame(rows)

# Add the parameter to MESSAGEix
msg.add_par("storage_self_discharge", storage_self_discharge_df)



In [51]:
msg.commit('')

In [52]:
caseName = msg.model + '__' + msg.scenario + '__v' + str(msg.version) +'ATR_test'
msg.solve(solve_options={'barcrossalg': '2','lpmethod':'1'}, case=caseName)

In [56]:
caseName

'MESSAGEix_CA__hydrogen_share__v65ATR_test'

In [53]:
from message_ix.report import Reporter
rep = Reporter.from_scenario(msg)
report = rep.get('message::default') # in fact, only in = input x ACT, out = output x ACT is needed

input: mixed units ['GWa', '-', '%'] discarded
emission_factor: mixed units ['tC', '???', 'kg/kWa', '-'] discarded


In [54]:
import numpy as np
min_year = 2020
max_year = 2060

report_df = report.timeseries()
report_df.reset_index(inplace=True)
report_df.columns = report_df.columns.astype(str)
report_df.columns = report_df.columns.str.title()

# Plotted years
years = msg.set('year').tolist()
years = [str(x) for x in years if x >= min_year and x <= max_year]

# Create data for Sankey diagrams and plot it using plotly
report_df.fillna(0, inplace = True)

# Convert from GWa to TWh (*8.76)
c = report_df.select_dtypes(include=[np.number]) * 8.76
report_df[c.columns] = c

# Drop not necessary years
report_df.columns = report_df.columns.astype(str)
report_df = report_df.drop(report_df.columns.to_series()['2020': str(min_year)],
                        axis=1)

# Identify input and output variables and its flows (provided within the Reporter: in = input x ACT, out = output x ACT)
in_flows = report_df[report_df['Variable'].str.startswith("in|")]
out_flows = report_df[report_df['Variable'].str.startswith("out|")]
all_flows = pd.concat([out_flows, in_flows])

report_df["Variable"] = report_df["Variable"].str.replace('emis', 'Emissions', regex = False)
#report_df["Variable"] = report_df["Variable"].str.replace('in|final|', 'Final Energy|', regex = False)
report_df["Variable"] = report_df["Variable"].str.replace('out|final|', 'Final Energy|', regex = False)
#report_df["Variable"] = report_df["Variable"].str.replace('in|primary|', 'Primary Energy|', regex = False)
report_df["Variable"] = report_df["Variable"].str.replace('out|primary|', 'Primary Energy|', regex = False)
report_df["Variable"] = report_df["Variable"].str.replace('in|renewable|', 'Primary Energy|renewable|', regex = False)
report_df["Variable"] = report_df["Variable"].str.replace('out|renewable|', 'Primary Energy|renewable|', regex = False)
report_df["Variable"] = report_df["Variable"].str.replace('total om cost|', 'Total Costs|', regex = False)
report_df["Variable"] = report_df["Variable"].str.replace('out|useful|', 'Useful Energy|', regex = False)
report_df["Variable"] = report_df["Variable"].str.replace('in|useful|', 'Useful Energy|', regex = False)

In [55]:
report_df.to_csv(f'{caseName}.csv', index = False)

In [ ]:
msg = message_set_up("baseline")

msg.set